# 05 · Limpieza · demanda comercial del SIN (XM)

`DemaCome` es la demanda **comercial**: la que se liquida en el mercado, que
incluye las pérdidas de la red. Por eso debería quedar siempre algo por encima
de la demanda real (`DemaReal`).

Lo que se encontró al diagnosticarla:

- serie completa, sin huecos ni duplicados;
- ~156 atípicos estacionales, casi todos en los dos últimos días publicados,
  que llegan parciales (el mismo defecto que la demanda real);
- **4 horas en las que la comercial queda por debajo de la real**, lo que no
  cuadra entre dos series de la misma fuente.

Se limpia con el mismo `limpiar()` que la demanda real, y además se marca la
incoherencia con la real. Marcar, no corregir: no hay forma de saber cuál de
las dos está mal.

In [1]:
import sys, warnings
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS = RAIZ / "datasets"
print("raiz     :", RAIZ)
print("datasets :", DATASETS, "->", "existe" if DATASETS.exists() else "FALTA")

raiz     : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER
datasets : C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets -> existe


In [2]:
from ingesta.normalizar import xm_cliente_a_esquema_comun
com = xm_cliente_a_esquema_comun(pd.read_csv(DATASETS / "xm" / "xm_demacome_sistema.csv", parse_dates=["timestamp"]))
real = xm_cliente_a_esquema_comun(pd.read_csv(DATASETS / "xm" / "xm_demareal_sistema.csv", parse_dates=["timestamp"]))
print(f"comercial: {len(com):,} filas · {com.fecha_hora.min()} .. {com.fecha_hora.max()}")
print(f"real     : {len(real):,} filas")

comercial: 49,824 filas · 2021-01-01 00:00:00-05:00 .. 2026-09-07 23:00:00-05:00
real     : 49,824 filas


## Diagnóstico

In [3]:
from calidad.diagnostico import diagnosticar, resumen
informe = diagnosticar(com)
print(resumen(informe))

DIAGNOSTICO DE CALIDAD
Generado    : 2026-09-11T04:58:13
Forma       : agregada (1.0 filas/hora)
Columnas    : tiempo=fecha_hora  valor=valor_kwh
Filas       : 49,824

--------------------------------------------------------------------------
COMPLETITUD TEMPORAL
--------------------------------------------------------------------------
Rango           : 2021-01-01 00:00:00-05:00 .. 2026-09-07 23:00:00-05:00
Horas           : 49,824 de 49,824 esperadas (100.0%)
Huecos          : ninguno
Timestamps rep. : 0
Dias != 24 h    : ninguno

--------------------------------------------------------------------------
VALORES
--------------------------------------------------------------------------
Descriptivos    : n=49,824  media=9,251,763.7  mediana=9,299,128.1
                  min=1,647,657.9  max=12,832,265.0
Nulos           : en 1 columnas
    version                    49,824  (100.0%)
Ceros/negativos : 0 ceros, 0 negativos
Atipicos IQR    : 48 (0.0963%)  fuera de [5,431,889, 13,056,431]


## Limpieza

In [4]:
from limpieza.limpiar import limpiar, procedencia, resumen as resumen_limpieza
limpio, registro = limpiar(com)
print(resumen_limpieza(registro))

REGISTRO DE LIMPIEZA
Momento : 2026-09-11T04:58:14
Filas   : 49,824 -> 49,824

[normalizar_esquema]  0 filas afectadas
  criterio: nombres a snake_case; marca de tiempo localizada en UTC-5; valor a float64
  columna_tiempo: fecha_hora
  columna_valor: valor_kwh
  zona_horaria: America/Bogota
  zona_ya_presente: True
  columnas_normalizadas_a_texto: ['fuente', 'metrica', 'entidad', 'version']

[deduplicar]  0 filas afectadas
  criterio: una fila por fecha_hora, conservando la ultima segun el orden de llegada
  filas_por_marca_antes: 1.0

[completar_rejilla]  0 filas afectadas
  criterio: rejilla horaria completa; imputacion 'temporal' solo para huecos de hasta 3 h; sin extrapolar en los extremos
  metodo_imputacion: temporal
  max_horas_interpolacion: 3
  columnas_constantes_propagadas: ['entidad', 'fuente', 'metrica']

[marcar_atipicos]  268 filas afectadas
  criterio: atipico = |z modificada CAUSAL frente a (dia de semana, hora)| > 3.5, calculada solo con observaciones anteriores; ati

## Coherencia con la demanda real

La comercial incluye pérdidas, así que el cociente comercial / real debería
estar siempre por encima de 1. Las horas en que no lo está se marcan en
`menor_que_real`.

In [5]:
par = limpio.merge(
    real[["fecha_hora", "valor_kwh"]].rename(columns={"valor_kwh": "real_kwh"}),
    on="fecha_hora", how="left", validate="one_to_one",
)
assert len(par) == len(limpio), "el merge cambió el número de filas"

cociente = par.valor_kwh / par.real_kwh
limpio["menor_que_real"] = (par.valor_kwh < par.real_kwh).to_numpy()

registro["operaciones"].append({
    "operacion": "marcar_incoherencia_con_real",
    "criterio": "menor_que_real = demanda comercial < demanda real en la misma hora; marcado, sin corregir",
    "filas_afectadas": int(limpio.menor_que_real.sum()),
    "detalle": {
        "cociente_mediana": round(float(cociente.median()), 4),
        "cociente_p1": round(float(cociente.quantile(.01)), 4),
        "cociente_p99": round(float(cociente.quantile(.99)), 4),
    },
})

print("cociente comercial / real:")
print(cociente.describe(percentiles=[.01, .5, .99]).round(4).to_string())
print()
print("horas marcadas como menor_que_real:", int(limpio.menor_que_real.sum()))
par.assign(cociente=cociente.round(4))[limpio.menor_que_real.to_numpy()][["fecha_hora", "valor_kwh", "real_kwh", "cociente"]]

cociente comercial / real:
count    49824.0000
mean         1.0154
std          0.0036
min          0.9940
1%           1.0070
50%          1.0152
99%          1.0245
max          1.0470

horas marcadas como menor_que_real: 4


,fecha_hora,valor_kwh,real_kwh,cociente
1020,2021-02-12 12:00:00-05:00,9482219.42,9504569.83,0.9976
1521,2021-03-05 09:00:00-05:00,8746672.52,8799817.21,0.9940
30129,2024-06-09 09:00:00-05:00,7940483.54,7942542.23,0.9997
34306,2024-11-30 10:00:00-05:00,10038685.06,10066659.41,0.9972


## Los últimos días publicados

Igual que en la demanda real, los dos últimos días llegan parciales. No hace
falta una marca nueva: el criterio causal de atípicos ya los señala.

In [6]:
diario = limpio.set_index("fecha_hora").resample("D").agg(
    gwh=("valor_kwh", lambda s: s.sum() / 1e6), atipicos=("atipico", "sum")
)
diario.tail(7).round(1)

,gwh,atipicos
fecha_hora,,
2026-09-01 00:00:00-05:00,260.2,0
2026-09-02 00:00:00-05:00,261.7,0
2026-09-03 00:00:00-05:00,265.2,0
2026-09-04 00:00:00-05:00,268.3,0
2026-09-05 00:00:00-05:00,253.0,0
2026-09-06 00:00:00-05:00,53.2,24
2026-09-07 00:00:00-05:00,54.3,24


## Guardar

In [7]:
import json
SALIDA = DATASETS / "limpios"
SALIDA.mkdir(parents=True, exist_ok=True)
limpio.to_parquet(SALIDA / "demanda_comercial.parquet", index=False)
with open(SALIDA / "registro_demanda_comercial.json", "w", encoding="utf-8") as f:
    json.dump(registro, f, ensure_ascii=False, indent=2, default=str)
print(f"{len(limpio):,} filas -> {SALIDA / 'demanda_comercial.parquet'}")

49,824 filas -> C:\Users\manue\OneDrive\Documentos\PROYECTOS PORTAFOLIO REAL\SIEM EXPLORER\datasets\limpios\demanda_comercial.parquet
